# 04 — Reranking: the spine

**Concept:** rerank a fixed candidate pool — with an LLM judge and with an
off-the-shelf cross-encoder.

## The prediction — written before running

> **nDCG@10 rises. recall@50 does not move — cannot move.**
>
> Reranking reorders a *fixed* candidate pool (dense top-50 here). A permutation
> cannot change which documents are in the pool, so recall at the pool size is
> structurally immune to anything the reranker believes. If recall@50 moves by any
> amount, the harness is broken — this prediction doubles as a self-check.

## Why this notebook is the spine

Modern eval stacks are increasingly LLMs all the way down: an LLM writes the
relevance labels, an LLM reranks the results, and the score goes up. Did retrieval
improve — or did the judge agree with itself?

The answer is not a better prompt. It's finding a **measurement the optimizer cannot
influence**. recall@pool-size is exactly that: the reranker can flatter its own
notion of relevance all it wants, but it physically cannot add a document to the
pool. NFCorpus gives us the luxury of *human* labels, so we can show both worlds in
one notebook: what the scores look like against independent truth, and what they
look like when the judge grades its own homework.

Two rerankers, same contract (a permutation of the pool):
- **LLM judge** — listwise 0–3 grader on headless Claude Code. Its grades are
  committed to `results/rerank/llm_grades.json`, so re-running this notebook needs
  no LLM and no API key.
- **`zerank-1-small`** — an off-the-shelf 1.7B cross-encoder (Apache-2.0). Scores
  committed to `results/rerank/zerank_scores.json`.

In [1]:
import numpy as np

from ragexp.data import load_nfcorpus
from ragexp.embed import Embedder
from ragexp.metrics import ndcg_at_k, paired_bootstrap
from ragexp.rerank import load_sweep, rerank_by_scores
from ragexp.retrieve import dense
from ragexp.runs import evaluate_run, metric_vector, save_scores, summarize

POOL = 50
corpus = load_nfcorpus()
qids = [q.query_id for q in corpus.queries]

emb = Embedder()
doc_matrix = emb.encode([d.full for d in corpus.docs])
pool = {q.query_id: dense(q.text, corpus.doc_ids, doc_matrix, emb)[:POOL]
        for q in corpus.queries}
print(f"pool: dense top-{POOL} for {len(pool)} queries")

pool: dense top-50 for 323 queries


In [2]:
llm_grades = load_sweep("llm_grades")        # {qid: {doc_id: 0..3}} — committed
zerank_scores = load_sweep("zerank_scores")  # {qid: {doc_id: score}} — committed
assert set(llm_grades) == set(pool) == set(zerank_scores), "sweeps must cover every query"

rankings = {
    "dense_pool": pool,  # the un-reranked pool IS the dense baseline at k <= 50
    "rerank_llm": {qid: rerank_by_scores(pool[qid], llm_grades[qid]) for qid in qids},
    "rerank_zerank": {qid: rerank_by_scores(pool[qid], zerank_scores[qid]) for qid in qids},
}
scores = {name: evaluate_run(corpus, r, recall_ks=(10, 50), ndcg_ks=(10,))
          for name, r in rankings.items()}
for name in ("rerank_llm", "rerank_zerank"):
    save_scores(name, scores[name])
for name, s in scores.items():
    print(f"{name:>14}: {({m: round(v, 4) for m, v in summarize(s).items()})}")

    dense_pool: {'recall@10': 0.155, 'recall@50': 0.2508, 'ndcg@10': 0.3167}
    rerank_llm: {'recall@10': 0.1865, 'recall@50': 0.2508, 'ndcg@10': 0.4052}
 rerank_zerank: {'recall@10': 0.1819, 'recall@50': 0.2508, 'ndcg@10': 0.3881}


## The structural check first

Before any bootstrap: recall@50 must be **bit-for-bit identical** across all three
systems, per query, because all three rank the same 50 documents. This is not a
statistical claim — it's an assertion. If it fails, nothing else in this notebook
can be trusted.

In [3]:
for name in ("rerank_llm", "rerank_zerank"):
    deltas = [abs(scores[name][qid]["recall@50"] - scores["dense_pool"][qid]["recall@50"])
              for qid in qids
              if scores[name][qid]["recall@50"] is not None]
    assert max(deltas) == 0.0, f"HARNESS BROKEN: recall@50 moved under {name}"
    print(f"{name:>14}: max |recall@50 delta| across {len(deltas)} queries = {max(deltas)}")
print("\nrecall@50 is frozen by construction — the self-check passes.")

    rerank_llm: max |recall@50 delta| across 323 queries = 0.0
 rerank_zerank: max |recall@50 delta| across 323 queries = 0.0

recall@50 is frozen by construction — the self-check passes.


## The real test — nDCG@10 against *human* labels

Order within the pool is what reranking can change, and nDCG@10 (scored against
NFCorpus's human qrels) is the order-sensitive instrument. recall@10 is also free to
move — top-10 *membership within the pool* changes even though pool membership can't
— and any movement there is real reordering, not judge self-flattery.

In [4]:
def compare(name_b):
    print(f"{name_b} vs dense_pool")
    for metric in ("ndcg@10", "recall@10"):
        a = metric_vector(scores["dense_pool"], metric, qids)
        b = metric_vector(scores[name_b], metric, qids)
        delta, (lo, hi), p = paired_bootstrap(a, b)
        sig = "  <-- significant" if (lo > 0 or hi < 0) else ""
        print(f"  {metric:>10}: delta {delta:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  p={p:.4f}{sig}")
    print()

compare("rerank_llm")
compare("rerank_zerank")

rerank_llm vs dense_pool
     ndcg@10: delta +0.0885  CI [+0.0715, +0.1057]  p=0.0000  <-- significant
   recall@10: delta +0.0315  CI [+0.0200, +0.0450]  p=0.0000  <-- significant

rerank_zerank vs dense_pool
     ndcg@10: delta +0.0714  CI [+0.0548, +0.0887]  p=0.0000  <-- significant
   recall@10: delta +0.0270  CI [+0.0149, +0.0406]  p=0.0000  <-- significant



## The judge grading its own homework

Now the dirty world: suppose we had no human labels and scored nDCG against the
**LLM judge's own grades**. The LLM rerank sorts by exactly those grades, so its
"nDCG" is 1.0 *by construction* — perfect score, zero information. The same
evaluation against human labels is the honest number. Side by side:

In [5]:
def ndcg_against(grades_by_q, ranking, k=10):
    vals = []
    for qid in qids:
        g = {d: int(v) for d, v in grades_by_q[qid].items()}
        v = ndcg_at_k(ranking[qid], g, k)
        if v is not None:
            vals.append(v)
    return float(np.mean(vals))

human = {qid: corpus.qrels[qid] for qid in qids}
print(f"{'system':>14} | {'nDCG@10 vs judge labels':>24} | {'nDCG@10 vs human labels':>24}")
for name in ("dense_pool", "rerank_llm"):
    j = ndcg_against(llm_grades, rankings[name])
    h = ndcg_against(human, rankings[name])
    print(f"{name:>14} | {j:>24.4f} | {h:>24.4f}")

        system |  nDCG@10 vs judge labels |  nDCG@10 vs human labels
    dense_pool |                   0.6367 |                   0.3167
    rerank_llm |                   1.0000 |                   0.4052


The left column says the reranker is spectacular. The left column is the judge
agreeing with itself — the reranked order *is* the judge's preference order, so
scoring it with the judge's labels is circular by construction. The right column is
what actually happened. Any eval pipeline where the labeler and the reranker share a
model family is somewhere between these two columns, and nothing in the nDCG math
warns you.

recall@50, meanwhile, printed the same number for every system — the measurement the
optimizer can't touch.

## Resolving the prediction

The prediction was **"nDCG rises, recall@50 does not move"** — and both halves held,
for both rerankers:

| | nDCG@10 | Δ vs dense (paired) | recall@50 |
|---|---|---|---|
| dense pool | 0.3167 | — | 0.2508 |
| LLM judge rerank | **0.4052** | +0.0885, CI [+0.072, +0.106], p ≈ 0 | 0.2508 — **frozen** |
| zerank-1-small | **0.3881** | +0.0714, CI [+0.055, +0.089], p ≈ 0 | 0.2508 — **frozen** |

1. **nDCG@10 rose, a lot, against human labels.** Reranking is the first method in
   this repo to beat the dense baseline — BM25 and RRF couldn't (notebook 03). A
   listwise LLM judge gained +0.089 nDCG@10; an off-the-shelf 1.7B cross-encoder
   captured most of that (+0.071) at zero marginal cost per run. recall@10 also rose
   (+0.032 / +0.027) — real reordering pushed relevant docs into the top 10.

2. **recall@50 did not move — to the last bit.** Not "no significant difference":
   the per-query maximum absolute delta is exactly 0.0 for both rerankers, because a
   permutation of a 50-doc pool cannot change what's in the pool. The harness
   self-check passes.

3. **The self-agreement table is the reason this notebook exists.** Scored against
   the judge's *own* grades, the LLM rerank gets nDCG@10 = 1.0000 — a perfect score
   containing zero information, since the reranked order *is* the judge's preference
   order. Scored against independent human labels, the honest number is 0.4052. Any
   pipeline where the labeler and the reranker share a model family lives somewhere
   between those columns — and the nDCG math will never warn you.

**The generalizing lesson:** when the thing being evaluated can influence the
metric, find a measurement it *cannot* influence and anchor there. Here that's
recall at pool size; in your system it's whatever the optimizer has no degrees of
freedom over. If that anchored metric ever moves, you haven't discovered an
improvement — you've discovered a bug.

Notebooks 01–04 are the stop-and-ship artifact. 05 (pooling vs summary) and 06
(learning-to-rank) are depth on top of it.